In [1]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
PRED = OUT + '/predictions'
import os, sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
os.makedirs(PRED, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
assert torch.cuda.is_available(), 'Switch runtime to GPU'
dev = 'cuda'

Mounted at /content/drive


In [2]:
X = torch.tensor(np.load(f'{DATA}/data/train_images.npy')).permute(0, 3, 1, 2).float().div(255)
XT = torch.tensor(np.load(f'{DATA}/data/test_images.npy')).permute(0, 3, 1, 2).float().div(255)
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
X, XT = ((X - mean) / std).to(dev), ((XT - mean) / std).to(dev)
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)
va = np.loadtxt(f'{DATA}/data/val_idx.txt', dtype=int)
noisy = torch.tensor(lab.noisy_label.values).to(dev)
clean = torch.tensor(lab.clean_label.values).to(dev)
tr_t, va_t = torch.tensor(tr).to(dev), torch.tensor(va).to(dev)
Xtr, Xva = X[tr_t], X[va_t]
ytr, yva_n, yva_c = noisy[tr_t], noisy[va_t], clean[va_t]
ok = ytr == clean[tr_t]
del X

In [3]:
def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def predict(T, feats=False):
    net.eval()
    P, Fs = [], []
    for k in range(0, len(T), 1000):
        with torch.autocast('cuda', dtype=torch.float16):
            P.append(F.softmax(net(T[k:k + 1000]).float(), 1))
        if feats:
            Fs.append(feat['f'].half())
    return torch.cat(P), (torch.cat(Fs) if feats else None)

In [4]:
torch.manual_seed(1); np.random.seed(1)
E, B = 60, 128
net = ResNet18(10).to(dev)
feat = {}
net.linear.register_forward_hook(lambda m, i, o: feat.__setitem__('f', i[0]))
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
FE = [10, 20, 30, 40, 50, 60]
P = np.zeros((E, len(Xtr), 10), dtype=np.float16)
Fsave = np.zeros((len(FE), len(Xtr), 512), dtype=np.float16)
TP = np.zeros((E, len(XT)), dtype=np.int8)
dyn = []
for ep in range(E):
    net.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            loss = F.cross_entropy(net(augment(Xtr[b])), ytr[b])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    e = ep + 1
    ptr, ftr = predict(Xtr, e in FE)
    P[ep] = ptr.cpu().numpy()
    if e in FE:
        Fsave[FE.index(e)] = ftr.cpu().numpy()
    pred = ptr.argmax(1)
    pv = predict(Xva)[0].argmax(1)
    TP[ep] = predict(XT)[0].argmax(1).cpu().numpy()
    row = [e, (pred[ok] == ytr[ok]).float().mean().item(), (pred[~ok] == ytr[~ok]).float().mean().item(),
           (pv == yva_n).float().mean().item(), (pv == yva_c).float().mean().item()]
    dyn.append(row)
    print(e, *[f'{x:.4f}' for x in row[1:]])

1 0.3713 0.1369 0.2768 0.3636
2 0.4126 0.1615 0.3154 0.3888
3 0.4815 0.1511 0.3552 0.4444
4 0.5506 0.1485 0.3858 0.5108
5 0.5586 0.1342 0.3888 0.5224
6 0.6313 0.1446 0.4210 0.5678
7 0.6538 0.1407 0.4312 0.5818
8 0.6790 0.1267 0.4486 0.6326
9 0.7414 0.1204 0.4796 0.6814
10 0.7589 0.1217 0.4938 0.6898
11 0.7395 0.1482 0.4902 0.6726
12 0.7337 0.1283 0.4780 0.6714
13 0.7584 0.1358 0.5032 0.6844
14 0.6364 0.1378 0.4312 0.5608
15 0.8177 0.1224 0.5264 0.7442
16 0.7883 0.1340 0.5092 0.7154
17 0.7562 0.1160 0.4814 0.6888
18 0.7828 0.1528 0.5126 0.6946
19 0.7965 0.1312 0.5170 0.7128
20 0.8269 0.1200 0.5180 0.7482
21 0.8058 0.1250 0.5076 0.7242
22 0.8093 0.1467 0.5234 0.7194
23 0.8230 0.1418 0.5334 0.7356
24 0.8577 0.1194 0.5414 0.7792
25 0.8327 0.1191 0.5132 0.7490
26 0.8528 0.1404 0.5372 0.7556
27 0.8646 0.1256 0.5394 0.7814
28 0.8407 0.1469 0.5324 0.7340
29 0.8619 0.1272 0.5330 0.7694
30 0.8618 0.1530 0.5434 0.7602
31 0.8662 0.1614 0.5464 0.7566
32 0.8830 0.1335 0.5456 0.7898
33 0.8739 0.1598 

In [5]:
np.save(f'{OUT}/run1_train_probs.npy', P)
np.save(f'{OUT}/run1_feats.npy', Fsave)
np.save(f'{OUT}/run1_test_preds.npy', TP)
D = pd.DataFrame(dyn, columns=['epoch', 'train_acc_correct', 'train_acc_mislabeled', 'val_acc_noisy', 'val_acc_clean'])
D.to_csv(f'{PRED}/t2_dynamics.csv', index=False)
es = int(D.val_acc_noisy.idxmax())
pd.DataFrame({'id': np.arange(len(XT)), 'pred': TP[-1]}).to_csv(f'{PRED}/t1_noisy_ce.csv', index=False)
pd.DataFrame({'id': np.arange(len(XT)), 'pred': TP[es]}).to_csv(f'{PRED}/t2_early_stop.csv', index=False)
print('early stop epoch', es + 1, ' val_noisy', D.val_acc_noisy[es], ' final val_noisy', D.val_acc_noisy.iloc[-1])

early stop epoch 37  val_noisy 0.5511999726295471  final val_noisy 0.49379998445510864


In [6]:
from sklearn.metrics import roc_auc_score
y, wrong = ytr.cpu().numpy(), (~ok).cpu().numpy()
k = wrong.sum()
p = P.astype(np.float32).mean(0)
pg = p[np.arange(len(y)), y]
o = p.copy(); o[np.arange(len(y)), y] = 0
S = np.log(o.max(1) + 1e-12) - np.log(pg + 1e-12)
print('precision@k', wrong[np.argsort(-S)[:k]].mean(), ' AUROC', roc_auc_score(wrong, S))

precision@k 0.8689601855431001  AUROC 0.9578417343133416


In [7]:
del P, Fsave, TP
torch.manual_seed(4); np.random.seed(4)
E, B, BETA, LAM = 60, 128, 0.7, 3.0
net = ResNet18(10).to(dev)
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
tgt = torch.zeros(len(Xtr), 10, device=dev)
P4 = np.zeros((E, len(Xtr), 10), dtype=np.float16)
for ep in range(E):
    net.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            out = net(augment(Xtr[b]))
        out = out.float()
        p = F.softmax(out, 1).clamp(1e-4, 1 - 1e-4)
        tgt[b] = BETA * tgt[b] + (1 - BETA) * (p / p.sum(1, keepdim=True)).detach()
        loss = F.cross_entropy(out, ytr[b]) + LAM * torch.log(1 - (tgt[b] * p).sum(1)).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    P4[ep] = predict(Xtr)[0].cpu().numpy()
    pv = predict(Xva)[0].argmax(1)
    print(ep + 1, f'{(pv == yva_n).float().mean().item():.4f} {(pv == yva_c).float().mean().item():.4f}')

1 0.3046 0.3828
2 0.3270 0.4316
3 0.3640 0.4610
4 0.3888 0.5076
5 0.4112 0.5498
6 0.4220 0.5628
7 0.4434 0.5980
8 0.4550 0.6208
9 0.4630 0.6504
10 0.4624 0.6354
11 0.4826 0.6664
12 0.4586 0.6242
13 0.5064 0.7080
14 0.5076 0.7216
15 0.5012 0.6934
16 0.4950 0.6806
17 0.4978 0.6876
18 0.5210 0.7390
19 0.5196 0.7398
20 0.5152 0.7366
21 0.5250 0.7238
22 0.5296 0.7632
23 0.5238 0.7528
24 0.5242 0.7550
25 0.5388 0.7796
26 0.5376 0.7630
27 0.5284 0.7740
28 0.5326 0.7728
29 0.5420 0.7868
30 0.5460 0.7944
31 0.5510 0.7878
32 0.5446 0.7850
33 0.5512 0.7940
34 0.5472 0.7918
35 0.5472 0.7916
36 0.5582 0.8162
37 0.5508 0.8010
38 0.5514 0.7934
39 0.5560 0.8088
40 0.5506 0.8092
41 0.5592 0.8074
42 0.5602 0.8250
43 0.5600 0.8130
44 0.5634 0.8300
45 0.5610 0.8308
46 0.5646 0.8242
47 0.5656 0.8338
48 0.5714 0.8404
49 0.5644 0.8312
50 0.5674 0.8268
51 0.5706 0.8448
52 0.5680 0.8348
53 0.5704 0.8436
54 0.5720 0.8438
55 0.5734 0.8458
56 0.5714 0.8438
57 0.5760 0.8478
58 0.5752 0.8454
59 0.5742 0.8464
60 0.5

In [8]:
np.save(f'{OUT}/run4_elr_train_probs.npy', P4)
from sklearn.metrics import roc_auc_score
y, wrong = ytr.cpu().numpy(), (~ok).cpu().numpy()
k = wrong.sum()
p = P4.astype(np.float32).mean(0)
pg = p[np.arange(len(y)), y]
o = p.copy(); o[np.arange(len(y)), y] = 0
S = np.log(o.max(1) + 1e-12) - np.log(pg + 1e-12)
print('precision@k', wrong[np.argsort(-S)[:k]].mean(), ' AUROC', roc_auc_score(wrong, S))

precision@k 0.865426031255177  AUROC 0.9485590933333907


In [3]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
PRED = OUT + '/predictions'
import os, sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
os.makedirs(PRED, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
assert torch.cuda.is_available(), 'Switch runtime to GPU'
dev = 'cuda'
X = torch.tensor(np.load(f'{DATA}/data/train_images.npy')).permute(0, 3, 1, 2).float().div(255)
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
X = ((X - mean) / std).to(dev)
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)
Xtr = X[torch.tensor(tr).to(dev)]
del X
y = lab.noisy_label.values[tr]
ytr = torch.tensor(y).to(dev)
wrong = y != lab.clean_label.values[tr]
k = wrong.sum()

def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def predict(T):
    net.eval()
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.cat([F.softmax(net(T[k:k + 1000]).float(), 1) for k in range(0, len(T), 1000)])

def sc(p):
    p = p.astype(np.float32)
    pg = p[np.arange(len(y)), y]
    o = p.copy(); o[np.arange(len(y)), y] = 0
    return np.log(o.max(1) + 1e-12) - np.log(pg + 1e-12)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
files = ['run1_train_probs.npy', 'run4_elr_train_probs.npy', 't3A_train_probs.npy', 't3B_train_probs.npy']
S = [sc(np.load(f'{OUT}/{f}').mean(0)) for f in files]
fused = sum(rankdata(s) for s in S)
r = rankdata(fused) / len(fused)
w = torch.tensor(np.clip((0.65 - r) / 0.25, 0, 1), dtype=torch.float32, device=dev)
print('mean weight', w.mean().item(), ' fraction at 0:', (w == 0).float().mean().item())

mean weight 0.5249888896942139  fraction at 0: 0.35002222657203674


In [5]:
torch.manual_seed(5); np.random.seed(5)
E, B = 60, 128
net = ResNet18(10).to(dev)
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
P5 = np.zeros((E, len(Xtr), 10), dtype=np.float16)
for ep in range(E):
    net.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            out = net(augment(Xtr[b]))
        l = F.cross_entropy(out.float(), ytr[b], reduction='none')
        loss = (l * w[b]).sum() / w[b].sum().clamp(min=1e-6)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    P5[ep] = predict(Xtr).cpu().numpy()
    print(ep + 1)
np.save(f'{OUT}/run5_clean_train_probs.npy', P5)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60


In [6]:
S5 = sc(P5.astype(np.float32).mean(0))
def chk(name, s):
    print(f'{name:12s} precision@k {wrong[np.argsort(-s)[:k]].mean():.4f}  AUROC {roc_auc_score(wrong, s):.4f}')
chk('run5 alone', S5)
final = sum(rankdata(s) for s in S + [S5])
chk('all 5 fused', final)
pd.DataFrame({'index': tr, 'score': final / final.max()}).to_csv(f'{PRED}/t3_noise_scores.csv', index=False)

run5 alone   precision@k 0.8807  AUROC 0.9629
all 5 fused  precision@k 0.8802  AUROC 0.9647


In [7]:
import os
print(os.path.exists('/content/drive/MyDrive/CV Bootcamp/outputs/predictions/t3_noise_scores.csv'))

True
